# Climate forward return — linear / ridge regression

**Option A:** Train on historical heatmap scenarios (Harvey, Uri, Ian, Camp Fire). Features:

| Feature | Source |
|---------|--------|
| `footprint_pct` | `facility_footprint_score × 100` (backend mispricing footprint) |
| `risk_score` | `_row_risk_score(predicted, gap, asset_class)` |
| `gap_pct` | predicted − actual position return in **anchor ±2d** event window |
| `hazard_intensity` | `footprint_score × scenario_severity × 20`, capped at 100 |
| `storm_phase` | Harvey/Ian: HURDAT track index at anchor ÷ (n−1); else `0.5` |

**Target:** **5 trading-day forward** position return (%) from first close on/after anchor (yfinance).

**Setup:** `cd backend`, activate venv, `pip install -r requirements.txt` (adds `pandas`, `scikit-learn`), then open this notebook with **working directory = `backend/`** so `import app` resolves.

This notebook does **not** import `heatmap_datasets` (avoids pulling `httpx` for a minimal analysis kernel). HURDAT parsing is inlined; scenarios match `HEATMAP_SCENARIOS` minus carbon.

In [1]:
from __future__ import annotations

import json
import re
import sys
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import yfinance as yf
from pandas.tseries.offsets import BDay
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import LeaveOneGroupOut

NB_PATH = Path.cwd()
for root in [NB_PATH, *NB_PATH.parents]:
    if (root / "app" / "services" / "heatmap_risk_intel.py").is_file():
        BACKEND_ROOT = root
        break
else:
    raise RuntimeError("cwd must be backend/ (directory containing app/)")

if str(BACKEND_ROOT) not in sys.path:
    sys.path.insert(0, str(BACKEND_ROOT))

from app.services.heatmap_risk_intel import (  # noqa: E402
    compute_all_scenario_dual_returns,
    facility_footprint_score,
    load_facility_rows,
    position_return_from_underlying,
    sensitivity_underlying_pct,
    _row_risk_score,
)

DATA_DIR = BACKEND_ROOT / "app" / "data"
CACHE_DIR = DATA_DIR / "cache"
HURDAT2_URL = "https://www.nhc.noaa.gov/data/hurdat/hurdat2-1851-2023-051124.txt"

# Same as heatmap_datasets.HEATMAP_SCENARIOS minus carbon (no real yfinance forward)
TRAIN_SCENARIOS: list[dict[str, str]] = [
    {"id": "harvey_2017", "label": "Hurricane Harvey — Aug 2017", "anchor_date": "2017-08-25", "hazard": "hurricane"},
    {"id": "uri_2021", "label": "Winter Storm Uri — Feb 2021", "anchor_date": "2021-02-15", "hazard": "freeze"},
    {"id": "ian_2022", "label": "Hurricane Ian — Sep 2022", "anchor_date": "2022-09-28", "hazard": "hurricane"},
    {"id": "camp_fire_2018", "label": "Camp Fire / CA wildfires — Nov 2018", "anchor_date": "2018-11-08", "hazard": "wildfire"},
]

SEVERITY_BY_ID: dict[str, int] = {
    "harvey_2017": 5,
    "uri_2021": 5,
    "ian_2022": 5,
    "camp_fire_2018": 4,
}

HURRICANE_SCENARIO_IDS = frozenset({"harvey_2017", "ian_2022"})
STORM_ID_BY_SCENARIO = {"harvey_2017": "AL092017", "ian_2022": "AL092022"}

_STORM_HEADER_RE = re.compile(r"^([A-Z]{2}\d{6}),\s*")


def _parse_hurdat_lat_lon(lat_raw: str, lon_raw: str) -> tuple[float, float] | None:
    lat_raw = lat_raw.strip().upper()
    lon_raw = lon_raw.strip().upper()
    if len(lat_raw) < 2 or len(lon_raw) < 2:
        return None
    lat_dir, lon_dir = lat_raw[-1], lon_raw[-1]
    if lat_dir not in "NS" or lon_dir not in "EW":
        return None
    try:
        lat = float(lat_raw[:-1]) * (1 if lat_dir == "N" else -1)
        lon = float(lon_raw[:-1]) * (-1 if lon_dir == "W" else 1)
    except ValueError:
        return None
    return lat, lon


def parse_hurdat_storms(raw_text: str, wanted_ids: tuple[str, ...]) -> dict[str, list[dict[str, Any]]]:
    wanted = set(wanted_ids)
    lines = raw_text.splitlines()
    out: dict[str, list[dict[str, Any]]] = {sid: [] for sid in wanted}
    current: str | None = None
    for line in lines:
        line = line.strip()
        if not line:
            continue
        m = _STORM_HEADER_RE.match(line)
        if m:
            sid = m.group(1)
            current = sid if sid in wanted else None
            continue
        if current is None:
            continue
        parts = [p.strip() for p in line.split(",")]
        if not parts or not parts[0].isdigit() or len(parts) < 6:
            continue
        parsed = _parse_hurdat_lat_lon(parts[4], parts[5])
        if not parsed:
            continue
        lat, lon = parsed
        date_utc = ""
        dr = parts[0]
        if len(dr) == 8 and dr.isdigit():
            date_utc = f"{dr[:4]}-{dr[4:6]}-{dr[6:8]}"
        out[current].append({"lat": lat, "lng": lon, "date_utc": date_utc})
    return out


def load_hurdat_tracks() -> dict:
    p = CACHE_DIR / "hurdat2_atlantic.txt"
    if not p.is_file():
        print("HURDAT cache missing; storm_phase defaults for hurricanes.")
        print(f"Expected: {p}\nOr download: curl -L -o {p} {HURDAT2_URL}")
        return {}
    raw = p.read_text(encoding="utf-8", errors="replace")
    return parse_hurdat_storms(raw, ("AL092017", "AL092022"))


def storm_phase_for_row(scenario_id: str, anchor: str, tracks: dict) -> float:
    if scenario_id not in HURRICANE_SCENARIO_IDS:
        return 0.5
    hid = STORM_ID_BY_SCENARIO.get(scenario_id)
    if not hid or hid not in tracks or len(tracks[hid]) < 2:
        return 0.55
    pts = tracks[hid]

    def parse_d(s: str):
        if len(s) != 10:
            return None
        try:
            return datetime.strptime(s, "%Y-%m-%d").date()
        except ValueError:
            return None

    t0 = datetime.strptime(anchor, "%Y-%m-%d").date()
    best_i, best_abs = 0, 10_000
    for i, pt in enumerate(pts):
        d = parse_d(pt.get("date_utc") or "")
        if d is None:
            continue
        a = abs((d - t0).days)
        if a < best_abs:
            best_abs, best_i = a, i
    return float(best_i) / float(max(1, len(pts) - 1))


def forward_return_5d_position(symbol: str, direction: str, anchor: str) -> float | None:
    mid = pd.Timestamp(anchor)
    start, end = mid - BDay(10), mid + BDay(20)
    hist = yf.download(symbol, start=start, end=end, progress=False, auto_adjust=True)
    if hist is None or hist.empty:
        return None
    closes = hist["Close"].dropna()
    if getattr(closes, "ndim", 1) > 1:
        closes = closes.iloc[:, 0]
    idx = closes.index[closes.index.normalize() >= mid.normalize()]
    if len(idx) < 1:
        return None
    loc = closes.index.get_loc(idx[0])
    if isinstance(loc, slice):
        loc = loc.start
    i0 = int(loc)
    if i0 + 5 >= len(closes):
        return None
    p0, p5 = float(closes.iloc[i0]), float(closes.iloc[i0 + 5])
    if p0 == 0:
        return None
    und = (p5 / p0 - 1.0) * 100.0
    sign = 1.0 if str(direction).lower() == "long" else -1.0
    return float(und * sign)


FACILITIES = load_facility_rows()
PORTFOLIO = json.loads((DATA_DIR / "heatmap_portfolio.json").read_text(encoding="utf-8"))
POSITIONS: list[dict] = PORTFOLIO.get("positions", [])
SYMBOLS = [p["symbol"] for p in POSITIONS]

tracks = load_hurdat_tracks()
wide_ret, event_ret = compute_all_scenario_dual_returns(SYMBOLS, TRAIN_SCENARIOS)

rows: list[dict] = []
for sc in TRAIN_SCENARIOS:
    aid = sc["id"]
    hazard = str(sc.get("hazard", "hurricane"))
    anchor = sc["anchor_date"]
    severity = float(SEVERITY_BY_ID.get(aid, 3))
    phase = storm_phase_for_row(aid, anchor, tracks)
    per_evt = event_ret.get(aid, {})

    for p in POSITIONS:
        sym = p["symbol"]
        direction = str(p.get("direction", "long"))
        ac = str(p.get("asset_class", "equity"))
        ev = per_evt.get(sym)
        if ev is None:
            continue
        und_pred = sensitivity_underlying_pct(sym, hazard, FACILITIES, ac)
        pred_pos = position_return_from_underlying(und_pred, direction)
        und_act_pct = round(float(ev) * 100.0, 6)
        act_pos = position_return_from_underlying(und_act_pct, direction)
        gap = round(pred_pos - act_pos, 6)
        fp = facility_footprint_score(sym, hazard, FACILITIES)
        footprint_pct = round(min(100.0, fp * 100.0), 4)
        risk = _row_risk_score(pred_pos, gap, ac)
        haz_i = round(min(100.0, fp * severity * 20.0), 4)
        y = forward_return_5d_position(sym, direction, anchor)
        if y is None:
            continue
        rows.append(
            {
                "scenario_id": aid,
                "anchor_date": anchor,
                "symbol": sym,
                "direction": direction,
                "footprint_pct": footprint_pct,
                "risk_score": risk,
                "gap_pct": gap,
                "hazard_intensity": haz_i,
                "storm_phase": round(phase, 4),
                "target_fwd_5d_pos_pct": round(y, 4),
            }
        )

df = pd.DataFrame(rows)
print("Rows (scenario × symbol with valid 5d forward):", len(df))
if df.empty:
    raise SystemExit("No rows — check network/yfinance and symbols.")
display(df.head(12))

feature_cols = ["footprint_pct", "risk_score", "gap_pct", "hazard_intensity", "storm_phase"]
X = df[feature_cols].values
y = df["target_fwd_5d_pos_pct"].values
groups = df["scenario_id"].values


Rows (scenario × symbol with valid 5d forward): 24


,scenario_id,anchor_date,symbol,direction,footprint_pct,risk_score,gap_pct,hazard_intensity,storm_phase,target_fwd_5d_pos_pct
0,harvey_2017,2017-08-25,VLO,long,73.50,50,-5.923,73.50,0.5068,2.2610
1,harvey_2017,2017-08-25,NRG,long,98.00,45,2.062,98.00,0.5068,0.9357
2,harvey_2017,2017-08-25,EGP,long,72.00,51,-4.158,72.00,0.5068,3.0734
3,harvey_2017,2017-08-25,ALL,long,60.12,52,-3.985,60.12,0.5068,-2.0344
4,harvey_2017,2017-08-25,CL=F,long,100.00,47,-3.885,100.00,0.5068,-1.2116
5,harvey_2017,2017-08-25,NG=F,short,100.00,56,-9.230,100.00,0.5068,-6.1549
6,uri_2021,2021-02-15,VLO,long,67.50,44,-2.407,67.50,0.5000,13.2168
7,uri_2021,2021-02-15,NRG,long,90.00,74,-12.380,90.00,0.5000,-5.6911
8,uri_2021,2021-02-15,EGP,long,35.00,41,1.011,35.00,0.5000,-2.4108
9,uri_2021,2021-02-15,ALL,long,41.60,44,-2.431,41.60,0.5000,4.6507


In [2]:
# --- In-sample metrics (small N: interpret with caution) ---
lin = LinearRegression().fit(X, y)
ridge = Ridge(alpha=2.0).fit(X, y)

for name, model in [("LinearRegression", lin), ("Ridge(alpha=2)", ridge)]:
    pred = model.predict(X)
    mse = mean_squared_error(y, pred)
    rmse = float(np.sqrt(mse))
    mae = mean_absolute_error(y, pred)
    r2 = r2_score(y, pred)
    print(f"\n{name} — in-sample (all {len(y)} rows)")
    print(f"  MSE:  {mse:.4f}")
    print(f"  RMSE: {rmse:.4f}  (% points)")
    print(f"  MAE:  {mae:.4f}")
    print(f"  R²:   {r2:.4f}")

print("\nCoefficients (LinearRegression, feature order):")
for c, v in zip(feature_cols, lin.coef_):
    print(f"  {c:20s} {v:+.6f}")
print(f"  {'intercept':20s} {lin.intercept_:+.6f}")

logo = LeaveOneGroupOut()
cv_mse_lin: list[float] = []
cv_mse_ridge: list[float] = []
for train_ix, test_ix in logo.split(X, y, groups):
    lin_cv = LinearRegression().fit(X[train_ix], y[train_ix])
    rid_cv = Ridge(alpha=2.0).fit(X[train_ix], y[train_ix])
    cv_mse_lin.append(mean_squared_error(y[test_ix], lin_cv.predict(X[test_ix])))
    cv_mse_ridge.append(mean_squared_error(y[test_ix], rid_cv.predict(X[test_ix])))

print("\nLeave-one-scenario-out CV (test = one full scenario at a time)")
print(f"  LinearRegression  mean MSE: {float(np.mean(cv_mse_lin)):.4f}  per-fold: {[round(x,4) for x in cv_mse_lin]}")
print(f"  Ridge(2)          mean MSE: {float(np.mean(cv_mse_ridge)):.4f}  per-fold: {[round(x,4) for x in cv_mse_ridge]}")

# --- Example recommendation line (synthetic features) ---
demo = np.array([[45.0, 68.0, -3.2, 55.0, 0.72]])
pred_demo = float(lin.predict(demo)[0])
tape_demo = 1.09
implied = tape_demo - pred_demo
print("\n--- Example copy (synthetic feature row) ---")
print(f"Model predicts {pred_demo:+.2f}% over next 5 sessions given cone-style features. ")
print(f"Current tape +{tape_demo:.2f}%. Implied alpha vs model: {implied:+.2f}%.")



LinearRegression — in-sample (all 24 rows)
  MSE:  15.5600
  RMSE: 3.9446  (% points)
  MAE:  3.1818
  R²:   0.5639

Ridge(alpha=2) — in-sample (all 24 rows)
  MSE:  15.6316
  RMSE: 3.9537  (% points)
  MAE:  3.2060
  R²:   0.5619

Coefficients (LinearRegression, feature order):
  footprint_pct        -0.571315
  risk_score           -0.598955
  gap_pct              -1.063329
  hazard_intensity     +0.565164
  storm_phase          +11.188090
  intercept            +21.939446

Leave-one-scenario-out CV (test = one full scenario at a time)
  LinearRegression  mean MSE: 502.7816  per-fold: [60.0607, 42.4617, 1775.6254, 132.9785]
  Ridge(2)          mean MSE: 46.0499  per-fold: [63.4474, 32.745, 9.3798, 78.6275]

--- Example copy (synthetic feature row) ---
Model predicts -1.96% over next 5 sessions given cone-style features. 
Current tape +1.09%. Implied alpha vs model: +3.05%.


## Limitations

- **Tiny N** (4 scenarios × 6 names = 24 rows if all fetch): treat coefficients as **illustrative**; prefer **LOOSO MSE** over in-sample R².
- **Leakage:** `gap_pct` uses returns in anchor **±2d**; target starts at anchor — not a clean causal split.
- **footprint_pct** is regional **footprint score**, not live map cone overlap.

Expand events, walk-forward validation, and stronger regularization for production.